<a href="https://colab.research.google.com/github/hristinagjorgjievska/Natural-Language-Processing/blob/master/lab2-cetvrtok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('sarcasm.txt', sep='\t')

In [ ]:
data

,Style 1,Style 2
0,The only thing I got from college is a caffein...,"College is really difficult, expensive, tiring..."
1,Remember the hundred emails from companies whe...,"I, at the bare minimum, wish companies actuall..."
2,Today my pop-pop told me I was not “forced” to...,"Today my pop-pop told me I was not ""forced"" to..."
3,@VolphanCarol @littlewhitty @mysticalmanatee I...,I would say Ted Cruz is an asshole and doesn’t...
4,Why would Alexa's recipe for Yorkshire pudding...,Great recipe from Alexa
...,...,...
688,"wow, ok, unfollowing now. Was a big fan of put...","I enjoy dancing the hokey cokey, in particular..."
689,Only posting selfie’s?? On YOUR own Instagram ...,It's my Instagram for a reason why wouldn't it...
690,do people with clear skin feel accomplished?? ...,people with clear skin are more comfortable in...
691,Update: holding hands with your mom and walkin...,Holding hands with your parent while walking a...


In [ ]:
train_size = int(0.7 * len(data))
val_size = int(0.15 * len(data))

In [ ]:
train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

In [18]:
from sentence_transformers import SentenceTransformer, util

In [19]:
train_sarcasm = train_data['Style 1'].values.tolist()
train_sincere = train_data['Style 2'].values.tolist()

val_sarcasm = val_data['Style 1'].values.tolist()
val_sincere = val_data['Style 2'].values.tolist()

test_sarcasm = test_data['Style 1'].values.tolist()
test_sincere = test_data['Style 2'].values.tolist()

In [20]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
train_val_sincere = train_sincere + val_sincere
train_val_sincere_emb = embedding_model.encode(train_val_sincere, batch_size=64)
test_emb = embedding_model.encode(test_sarcasm, batch_size=64)

In [33]:
train_val_sincere_emb.shape, test_emb.shape

((588, 384), (105, 384))

In [34]:
test_hits = util.semantic_search(test_emb, train_val_sincere_emb, top_k=2)

In [35]:
test_hits[0]

[{'corpus_id': 317, 'score': 0.3436564803123474},
 {'corpus_id': 578, 'score': 0.32354575395584106}]

In [36]:
!pip install evaluate
!pip install bert_score

In [37]:
from evaluate import load

In [38]:
bertscore = load('bertscore')

In [39]:
prediction1 = []
prediction2 = []

for hits in test_hits:
  prediction1.append(train_val_sincere[hits[0]['corpus_id']])
  prediction2.append(train_val_sincere[hits[1]['corpus_id']])


In [42]:
score1 = bertscore.compute(predictions=prediction1, references=test_sincere, model_type='distilroberta-base')['f1']

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
score2 = bertscore.compute(predictions=prediction2, references=test_sincere, model_type='distilroberta-base')['f1']

In [48]:
results = pd.DataFrame({'sarcastic': test_sarcasm, 'score1': score1, 'score2': score2})
results

,sarcastic,score1,score2
0,‘It’s only a game’ 🥴🥴,0.869279,0.857147
1,I love to pick my modules on a first come firs...,0.865570,0.869066
2,"Oh good, it's time for the bi-monthly random p...",0.865828,0.854797
3,Wish I was the bitch that Elijah hewson fuckin...,0.844516,0.837762
4,Might just fuck around and take some horse dew...,0.852116,0.862320
...,...,...,...
100,"wow, ok, unfollowing now. Was a big fan of put...",0.857658,0.900029
101,Only posting selfie’s?? On YOUR own Instagram ...,0.868990,0.872774
102,do people with clear skin feel accomplished?? ...,0.866958,0.865671
103,Update: holding hands with your mom and walkin...,0.877191,0.855760


In [49]:
train_val_sarcasm = train_sarcasm + val_sarcasm

In [50]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [51]:
model_name = 't5-large'

In [52]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.95GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
prompt_given_sentences = []

for i in range(len(test_sarcasm)):
    examples = ''
    for hit in test_hits[i]:
        j = hit['corpus_id']
        examples += f'Input: {train_val_sarcasm[j]} Output: {train_val_sincere[j]}\n'

    prompt = f'Examples:\n{examples}Rewrite the following sarcastic text as a neutral text: {test_sarcasm[i]} Output: '

    tokens = tokenizer(prompt, return_tensors='pt')
    output_ids = model.generate(tokens.input_ids, max_new_tokens=50)
    prompt_given_sentences.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [ ]:
score_prompt = bertscore.compute(predictions=prompt_given_sentences, references=test_sincere, model_type='distilroberta-base')['f1']

In [ ]:
results['score_prompt'] = score_prompt
results